In [6]:
from pathlib import Path
import pandas as pd

# ==========================================================
# Load and Prepare Data
# ==========================================================
# Load cleaned datasets from processed folder
processed_dir = Path("data/processed")
sales_clean = pd.read_csv(processed_dir / "sales_clean.csv")
future_clean = pd.read_csv(processed_dir / "future_clean.csv")

# Convert date columns to datetime
sales_clean["date"] = pd.to_datetime(sales_clean["date"])
future_clean["date"] = pd.to_datetime(future_clean["date"])



C:\Users\angel\AppData\Local\Temp\ipykernel_11800\67329991.py:9: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_clean = pd.read_csv(processed_dir / "sales_clean.csv")


In [9]:
# ==========================================================
# Feature Engineering
# ==========================================================
# Machine learning models cannot directly understand dates.
# We transform the date into calendar numerical features
# to help the model learn weekly and seasonal sales patterns.
# ==========================================================

# Copy datasets to avoid modifying the cleaned data
sales_fe = sales_clean.copy()
future_fe = future_clean.copy()

In [12]:
# ==========================================================
#  Calendar Features
# ==========================================================
# Features:
# month        : Month of the year (1-12)
# quarter      : Quarter of the year (1-4)
# week_of_year : ISO week number (1-52)
# day_of_week  : Day of the week (Monday=0, Sunday=6)
# day_of_month : Day of the month (1-31)
# is_weekend   : Weekend indicator (0=Weekday, 1=Weekend)

calendar_features = [
    "month",
    "quarter",
    "week_of_year",
    "day_of_week",
    "day_of_month",
    "is_weekend"
]

# ---------- Training ----------
sales_fe["month"] = sales_fe["date"].dt.month

sales_fe["quarter"] = sales_fe["date"].dt.quarter

sales_fe["week_of_year"] = (
    sales_fe["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

sales_fe["day_of_week"] = sales_fe["date"].dt.dayofweek

sales_fe["day_of_month"] = sales_fe["date"].dt.day

sales_fe["is_weekend"] = (
    sales_fe["day_of_week"] >= 5
).astype(int)


# ---------- Future ----------
future_fe["month"] = future_fe["date"].dt.month

future_fe["quarter"] = future_fe["date"].dt.quarter

future_fe["week_of_year"] = (
    future_fe["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

future_fe["day_of_week"] = future_fe["date"].dt.dayofweek

future_fe["day_of_month"] = future_fe["date"].dt.day

future_fe["is_weekend"] = (
    future_fe["day_of_week"] >= 5
).astype(int)

# ==========================================================
# Validate Calendar Features
# ==========================================================

print("Training data")
display(
    sales_fe[
        [
            "date",
            *calendar_features
        ]
    ].head()
)

print("Future data")
display(
    future_fe[
        [
            "date",
            *calendar_features
        ]
    ].head()
)

# ==========================================================
# Check Missing Values
# ==========================================================

print("Training")
print(
    sales_fe[calendar_features]
    .isna()
    .sum()
)

print("\nFuture")
print(
    future_fe[calendar_features]
    .isna()
    .sum()
)

Training data


,date,month,quarter,week_of_year,day_of_week,day_of_month,is_weekend
0,2015-07-19,7,3,29,6,19,1
1,2015-07-19,7,3,29,6,19,1
2,2015-07-19,7,3,29,6,19,1
3,2015-07-19,7,3,29,6,19,1
4,2015-07-19,7,3,29,6,19,1


Future data


,date,month,quarter,week_of_year,day_of_week,day_of_month,is_weekend
0,2015-09-17,9,3,38,3,17,0
1,2015-09-17,9,3,38,3,17,0
2,2015-09-17,9,3,38,3,17,0
3,2015-09-17,9,3,38,3,17,0
4,2015-09-17,9,3,38,3,17,0


Training
month           0
quarter         0
week_of_year    0
day_of_week     0
day_of_month    0
is_weekend      0
dtype: int64

Future
month           0
quarter         0
week_of_year    0
day_of_week     0
day_of_month    0
is_weekend      0
dtype: int64


In [34]:
# ==========================================================
# One-Hot Encoding
# ==========================================================
# Categorical features like store_type, assortment, and
# state_holiday are non-numeric and need to be converted
# into numeric representations. One-hot encoding creates
# binary columns for each category value.
#
# Features to encode:
# store_type   : Type of store (a, b, c, d)
# assortment   : Product assortment level (a, b, c)
# state_holiday: Public holiday (0, a, b, c)
#
# These categorical features provide important business
# context that helps machine learning models make better
# predictions about sales patterns.
# ==========================================================

# Identify one-hot encoded columns in sales_fe
encoded_columns_sales = [
    col for col in sales_fe.columns 
    if any(col.startswith(cat) for cat in ["store_type_", "assortment_", "state_holiday_"])
]

# Identify one-hot encoded columns in future_fe
encoded_columns_future = [
    col for col in future_fe.columns 
    if any(col.startswith(cat) for cat in ["store_type_", "assortment_", "state_holiday_"])
]

print("=== One-Hot Encoded Columns ===")
print(f"\nTraining set encoded columns ({len(encoded_columns_sales)}):")
print(encoded_columns_sales)

print(f"\nFuture set encoded columns ({len(encoded_columns_future)}):")
print(encoded_columns_future)

# ==========================================================
# Validate One-Hot Encoded Features
# ==========================================================

print("\n=== Training Data ===")
display(sales_fe[encoded_columns_sales].head())

print("\nTraining - Missing Values:")
print(sales_fe[encoded_columns_sales].isna().sum())

print("\n=== Future Data ===")
display(future_fe[encoded_columns_future].head())

print("\nFuture - Missing Values:")
print(future_fe[encoded_columns_future].isna().sum())

=== One-Hot Encoded Columns ===

Training set encoded columns (11):
['store_type_a', 'store_type_b', 'store_type_c', 'store_type_d', 'assortment_a', 'assortment_b', 'assortment_c', 'state_holiday_0', 'state_holiday_a', 'state_holiday_b', 'state_holiday_c']

Future set encoded columns (11):
['store_type_a', 'store_type_b', 'store_type_c', 'store_type_d', 'assortment_a', 'assortment_b', 'assortment_c', 'state_holiday_0', 'state_holiday_a', 'state_holiday_b', 'state_holiday_c']

=== Training Data ===


,store_type_a,store_type_b,store_type_c,store_type_d,assortment_a,assortment_b,assortment_c,state_holiday_0,state_holiday_a,state_holiday_b,state_holiday_c
0,False,False,True,False,True,False,False,True,False,False,False
1,False,False,True,False,True,False,False,True,False,False,False
2,False,False,True,False,True,False,False,True,False,False,False
3,False,False,True,False,True,False,False,True,False,False,False
4,False,False,True,False,True,False,False,True,False,False,False



Training - Missing Values:
store_type_a       0
store_type_b       0
store_type_c       0
store_type_d       0
assortment_a       0
assortment_b       0
assortment_c       0
state_holiday_0    0
state_holiday_a    0
state_holiday_b    0
state_holiday_c    0
dtype: int64

=== Future Data ===


,store_type_a,store_type_b,store_type_c,store_type_d,assortment_a,assortment_b,assortment_c,state_holiday_0,state_holiday_a,state_holiday_b,state_holiday_c
0,False,False,True,False,True,False,False,True,0,0,0
1,True,False,False,False,True,False,False,True,0,0,0
2,True,False,False,False,False,False,True,True,0,0,0
3,True,False,False,False,True,False,False,True,0,0,0
4,True,False,False,False,False,False,True,True,0,0,0



Future - Missing Values:
store_type_a       0
store_type_b       0
store_type_c       0
store_type_d       0
assortment_a       0
assortment_b       0
assortment_c       0
state_holiday_0    0
state_holiday_a    0
state_holiday_b    0
state_holiday_c    0
dtype: int64


In [22]:
# ==========================================================
#  Lag Features
# ==========================================================
# Lag features provide historical sales information so the
# model can learn temporal dependencies.
#
# Features:
# lag_1   : Sales from previous day
# lag_7   : Sales from same weekday last week
# lag_14  : Sales from two weeks ago
# lag_21  : Sales from three weeks ago
# lag_28  : Sales from four weeks ago
#
# Lag features are among the most important predictors for
# tree-based forecasting models.
# ==========================================================

# ==========================================================
# Generate Lag Features
# ==========================================================

# Important: sort by store and date first so each store's
# lag is computed from its own previous day, not from the
# previous row in the raw file order.
sales_fe = sales_fe.sort_values(["store_id", "date"]).reset_index(drop=True)

lag_features = [1, 7, 14, 21, 28]

for lag in lag_features:
    sales_fe[f"lag_{lag}"] = (
        sales_fe.groupby("store_id", sort=False)["sales"]
        .shift(lag)
    )

# ==========================================================
# Validate Lag Features
# ==========================================================

lag_columns = [f"lag_{lag}" for lag in lag_features]

display(
    sales_fe[
        [
            "store_id",
            "date",
            "sales",
            *lag_columns
        ]
    ].head(35)
)

# ==========================================================
# Check Missing Values
# ==========================================================

print(sales_fe[lag_columns].isna().sum())
print("\nNon-null lag_1 values:", sales_fe["lag_1"].notna().sum())

,store_id,date,sales,lag_1,lag_7,lag_14,lag_21,lag_28
0,store_1,2013-01-07,7176,NaN,NaN,NaN,NaN,NaN
1,store_1,2013-01-08,5580,7176.0,NaN,NaN,NaN,NaN
2,store_1,2013-01-09,5471,5580.0,NaN,NaN,NaN,NaN
3,store_1,2013-01-10,4892,5471.0,NaN,NaN,NaN,NaN
4,store_1,2013-01-11,4881,4892.0,NaN,NaN,NaN,NaN
5,store_1,2013-01-12,4952,4881.0,NaN,NaN,NaN,NaN
6,store_1,2013-01-13,0,4952.0,NaN,NaN,NaN,NaN
7,store_1,2013-01-14,4717,0.0,7176.0,NaN,NaN,NaN
8,store_1,2013-01-15,3900,4717.0,5580.0,NaN,NaN,NaN
9,store_1,2013-01-16,4008,3900.0,5471.0,NaN,NaN,NaN


lag_1       676
lag_7      4732
lag_14     9464
lag_21    14196
lag_28    18928
dtype: int64

Non-null lag_1 values: 623948


In [25]:
# ==========================================================
# Rolling Features: Mean and Std
# ==========================================================
# Lag features provide a single historical observation,
# while rolling features summarize recent sales behaviour.
#
# Features:
# rolling_mean_7  : Average sales over previous 7 days
# rolling_mean_14 : Average sales over previous 14 days
# rolling_mean_28 : Average sales over previous 28 days
#
# Rolling features help machine learning models capture
# short-term trends and smooth out daily fluctuations.
# ==========================================================

rolling_windows = [7, 14, 21, 28]

for window in rolling_windows:

    sales_fe[f"rolling_mean_{window}"] = (
        sales_fe
        .groupby("store_id")["sales"]
        .transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

    sales_fe[f"rolling_std_{window}"] = (
        sales_fe
        .groupby("store_id")["sales"]
        .transform(
            lambda x: x.shift(1).rolling(window).std()
        )
    )

#Due to Data leakage, we shift the rolling mean by 1 day to ensure that 
#the model does not have access to future information when making predictions.

# ==========================================================
# Validate Rolling Features
# ==========================================================

rolling_columns = []

for window in rolling_windows:
    rolling_columns.extend([
        f"rolling_mean_{window}",
        f"rolling_std_{window}"
    ])

store = "store_1"

display(
    sales_fe.loc[
        sales_fe["store_id"] == store,
        [
            "date",
            "sales",
            *rolling_columns
        ]
    ].head(35)
)

# ==========================================================
# Check Missing Values
# ==========================================================

print(sales_fe[rolling_columns].isna().sum())



,date,sales,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,rolling_mean_21,rolling_std_21,rolling_mean_28,rolling_std_28
0,2013-01-07,7176,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-01-08,5580,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013-01-09,5471,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2013-01-10,4892,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2013-01-11,4881,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2013-01-12,4952,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2013-01-13,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2013-01-14,4717,4707.428571,2225.689396,NaN,NaN,NaN,NaN,NaN,NaN
8,2013-01-15,3900,4356.142857,1947.844743,NaN,NaN,NaN,NaN,NaN,NaN
9,2013-01-16,4008,4116.142857,1874.016847,NaN,NaN,NaN,NaN,NaN,NaN


rolling_mean_7      4732
rolling_std_7       4732
rolling_mean_14     9464
rolling_std_14      9464
rolling_mean_21    14196
rolling_std_21     14196
rolling_mean_28    18928
rolling_std_28     18928
dtype: int64
